# Breast Cancer CAD System — Master Notebook

End-to-end pipeline: **Preprocessing -> YOLO26n (detection) -> U-Net (segmentation) -> ResNet50 (classification)**.

Every cell below calls into the reusable modules under `preprocessing/`, `yolo/`, `unet/`, `resnet/`, `inference/` — no pipeline logic is duplicated here.

**Important:** the training cells (Stage 2/3/4) are provided fully ready-to-run but are **not meant to be executed automatically** (e.g. via "Run All") — training each stage takes substantial time and should be run deliberately, one stage at a time, on your own hardware/schedule.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.config import load_config
from utils.seed import set_seed, get_device

config = load_config(str(REPO_ROOT / "config.yaml"))
# Uncomment to smoke-test the whole pipeline on a tiny subset before a real run:
# config = load_config(str(REPO_ROOT / "config.yaml"), overrides=str(REPO_ROOT / "configs/quick_run.yaml"))

set_seed(config["seed"], config["deterministic"])
device = get_device(config.get("device"))
print(f"Device: {device}")
print(f"Repo root: {REPO_ROOT}")


## Stage 1 — Preprocessing & Augmentation

Validates every image/mask pair, builds a stratified 70/15/15 split (content-duplicate-aware — see the
`duplicate_image_cross_class` check in `preprocessing/validation.py`), derives per-lesion YOLO bounding
boxes from the masks, and renders sanity-check visualizations. No model training happens in this stage —
safe to re-run any time.

In [ ]:
from preprocessing.pipeline import run_stage1

stage1_result = run_stage1(config)
print(stage1_result["report"])


In [ ]:
from IPython.display import Image as IPImage, display

results_dir = REPO_ROOT / config["paths"]["results"] / "preprocessing"
display(IPImage(filename=str(results_dir / "mask_sanity_check.png")))
display(IPImage(filename=str(results_dir / "bbox_overlay_check.png")))


In [ ]:
import random

import albumentations as A

from dataset.lesion_manifest import load_lesion_manifest
from preprocessing.mask_to_yolo import component_mask
from preprocessing.normalization import load_image_rgb
from preprocessing.roi_utils import crop_roi_pair
from preprocessing.augmentation import (
    build_yolo_augmentations,
    build_unet_augmentations,
    build_resnet_augmentations,
    visualize_augmentations,
)

train_lesions = load_lesion_manifest(REPO_ROOT / config["paths"]["splits_dir"] / "train_lesions.json")
sample_row = random.Random(config["seed"]).choice(train_lesions)
sample_image = load_image_rgb(sample_row["image_path"])
sample_mask = component_mask(sample_row["source_mask_path"], sample_row["component_label"])
roi_size = tuple(config["preprocessing"]["roi_size"])
roi_crop, mask_crop, _ = crop_roi_pair(
    sample_image, sample_mask, tuple(sample_row["bbox"]), pad_ratio=0.0, target_size=roi_size
)

# build_yolo_augmentations() requires bbox_params at call time; for a plain
# before/after preview (no bboxes to carry through) drop the Resize step and
# reuse the same image-level transform list without bbox_params attached.
yolo_transforms = build_yolo_augmentations(config["yolo"]["imgsz"]).transforms
yolo_preview = A.Compose([t for t in yolo_transforms if not isinstance(t, A.Resize)])

print("YOLO-stage augmentations (full safe + cautious set):")
visualize_augmentations(sample_image, yolo_preview, n_samples=5)

print("U-Net-stage augmentations (safe + cautious + tiny elastic):")
visualize_augmentations(roi_crop, build_unet_augmentations(roi_size), n_samples=5, mask=mask_crop)

print("ResNet-stage augmentations (conservative safe set only):")
visualize_augmentations(roi_crop, build_resnet_augmentations(roi_size), n_samples=5)


## Stage 2 — YOLO26n Detection

Builds the Ultralytics-format dataset (images + YOLO labels + `data.yaml`) from the Stage 1 split, then
trains YOLO26n to localize the tumor. The dataset-build cell is safe to run any time; **the training
cell trains a real model and should be run deliberately, not as part of "Run All".**

In [ ]:
from dataset.splits import load_split
from yolo.dataset_builder import build_yolo_dataset

splits = {name: load_split(REPO_ROOT / config["paths"]["splits_dir"], name) for name in ("train", "val", "test")}
data_yaml = build_yolo_dataset(
    splits,
    REPO_ROOT / config["paths"]["yolo_dataset_dir"],
    min_area_px=config["preprocessing"]["min_mask_area_px"],
    bbox_pad_ratio=config["preprocessing"]["bbox_pad_ratio"],
)
print(f"YOLO dataset ready at {data_yaml}")


In [ ]:
# TRAINING CELL — trains a real model. Run manually and deliberately, not via "Run All".
from yolo.train import train_yolo

train_yolo(config, data_yaml)


In [ ]:
# Evaluation — run after the training cell above has produced a checkpoint.
from yolo.evaluate import evaluate_yolo, compute_mean_iou

yolo_weights = REPO_ROOT / config["yolo"]["project"] / config["yolo"]["name"] / "weights" / "best.pt"
yolo_metrics = evaluate_yolo(yolo_weights, data_yaml, imgsz=config["yolo"]["imgsz"])

test_lesions = load_lesion_manifest(REPO_ROOT / config["paths"]["splits_dir"] / "test_lesions.json")
iou_metrics = compute_mean_iou(yolo_weights, test_lesions, imgsz=config["yolo"]["imgsz"])
print(yolo_metrics, iou_metrics)


## Stage 3 — U-Net Segmentation

Trains a U-Net (default: ResNet34 encoder, ImageNet-pretrained) on ROI crops derived from the
ground-truth lesion boxes. To compare architectures, reload `config` with one of the
`configs/unet_*.yaml` overrides (attention gates, EfficientNet-B0, U-Net++) before running the
training cell. **Trains a real model — run manually, not via "Run All".**

In [ ]:
# TRAINING CELL — trains a real model. Run manually and deliberately, not via "Run All".
from unet.train import train_unet

train_unet(config)


In [ ]:
# Evaluation — run after the training cell above has produced a checkpoint.
from unet.evaluate import evaluate_unet

unet_checkpoint = REPO_ROOT / config["unet"]["checkpoint_dir"] / config["unet"]["checkpoint_name"]
unet_metrics = evaluate_unet(config, unet_checkpoint)
print(unet_metrics)


## Stage 4 — ResNet50 Classification

Two-phase transfer learning (linear-probe warmup, then full fine-tuning) on the masked tumor ROI,
using ground-truth lesion masks so this stage trains independently of Stage 3's own predictions.
**Trains a real model — run manually, not via "Run All".**

In [ ]:
# TRAINING CELL — trains a real model. Run manually and deliberately, not via "Run All".
from resnet.train import train_resnet

train_resnet(config)


In [ ]:
# Evaluation — run after the training cell above has produced a checkpoint.
from resnet.evaluate import evaluate_resnet

resnet_checkpoint = REPO_ROOT / config["resnet"]["checkpoint_dir"] / config["resnet"]["checkpoint_name"]
resnet_metrics = evaluate_resnet(config, resnet_checkpoint)
print(resnet_metrics)


## End-to-end cascade evaluation

Chains the *real* trained YOLO26n -> U-Net -> ResNet50 (not ground-truth-fed) over the test set, to
measure how much error propagates through the full cascade versus each stage's isolated evaluation
above. Requires all three checkpoints from Stages 2-4 to already exist.

In [ ]:
from evaluation.cascade import evaluate_cascade

cascade_metrics = evaluate_cascade(config, split="test")
print(cascade_metrics)


## Single-image inference demo

Runs the full cascade on one raw ultrasound image and visualizes every intermediate output
(detection, ROI, segmentation overlay, classification). Requires all three checkpoints to exist.

In [ ]:
from inference.pipeline import build_pipeline
from inference.visualize import plot_case_result

test_rows = load_split(REPO_ROOT / config["paths"]["splits_dir"], "test")
sample_image_path = test_rows[0]["image_path"]

pipeline = build_pipeline(config)
case_result = pipeline.run(sample_image_path)
plot_case_result(case_result)
print(f"Predicted: {case_result.predicted_class} ({case_result.confidence:.1%})")


## Stage 5 — Clinician-in-the-loop demo

For interactive, physician-reviewed inference with accept/reject controls at every stage, run:

```bash
streamlit run app/app.py
```

The app reuses `inference/pipeline.py` and `preprocessing/roi_utils.py` — the same modules used above —
so its behavior matches this notebook's cascade exactly, with physician review layered on top.